# S01 — Streaming Fundamentals

**Time: about 50 minutes.**
Covers: Streaming Fundamentals

### What you will be able to do afterwards

- Explain what a micro-batch is and what the checkpoint stores.
- Choose between `availableNow` and `processingTime` triggers and say why.
- Pick the right output mode for a query, and know which ones are illegal when.
- Kill a stream mid-run and restart it without losing or duplicating data.

### Why there are no files in this notebook

You will use Spark's built-in **rate source**: a stream that emits a fixed number of rows
per second, forever, with no files, no volume and no external system. `helpers.event_stream`
projects it into the same clickstream shape used in the rest of the track.

That means when something goes wrong in this notebook, it is your streaming code — not a
file that failed to upload or a schema that drifted. Files arrive in S02.

**Compute:** serverless. Keep triggers short and always stop your queries.

In [ ]:
from pyspark.sql import functions as F
from helpers import utils, event_stream, streaming_utils

cfg = utils.get_configs("rate_events")
rate_table = cfg["table_bronze"]
rate_checkpoint = cfg["checkpoint_path"]

agg_cfg = utils.get_configs("rate_events_by_type")
agg_table = agg_cfg["table_silver"]
agg_checkpoint = agg_cfg["checkpoint_path"]

spark = utils.spark
print(rate_table, rate_checkpoint, agg_table, sep="\n")

## Step 1 — Look at a stream before writing it anywhere

**TO DO**

1. Call `event_stream.rate_events(rows_per_second=20)` and print the schema.
2. Confirm it is a streaming DataFrame: `df.isStreaming`.
3. Try `df.count()`. Read the error carefully — it is one of the most useful errors in
   Structured Streaming.

> **Question:** `df.count()` fails on a streaming DataFrame. Explain why in terms of what a
> stream *is*, not in terms of the API.

In [ ]:
# TO DO: build the rate stream, print its schema, check isStreaming


# TO DO: try .count() and read the error

## Step 2 — Write it, with `availableNow`

`trigger(availableNow=True)` processes everything currently available and then stops. For
the rate source that means one bounded chunk, which makes it perfect for a first run.

**TO DO**

1. Write the rate stream to `rate_table` in append mode, with `rate_checkpoint` as the
   checkpoint location.
2. Use `trigger(availableNow=True)` and `awaitTermination()`.
3. Query the table and count the rows.
4. Look at `DESCRIBE HISTORY` — how does a streaming write show up compared to a batch one?

**Tip:** if you need to start over, `utils.reset_path(rate_checkpoint)` deletes the
checkpoint, and you should drop the table too. A checkpoint that has moved on from an empty
table is the single most common way to get a stream that "does nothing".

In [ ]:
# TO DO: write the stream with availableNow and wait for it


# TO DO: count the rows and inspect the history

## Step 3 — Switch to `processingTime` and watch it iterate

**TO DO**

1. Start the same write again, this time with `trigger(processingTime="5 seconds")` and
   **without** `awaitTermination()` — this query will not stop on its own.
2. Use `streaming_utils.await_batches(query, batches=4)` to block until four micro-batches
   have completed.
3. Call `streaming_utils.show_progress(query)` and look at `input_rows` and
   `batch_duration_ms` per batch.
4. Stop the query with `query.stop()`.

**Do not skip step 4.** A `processingTime` query runs until stopped, and on serverless that
means it keeps consuming compute after you close the notebook.

> **Questions:**
> - You reused the same checkpoint. Did the new run reprocess the rows from Step 2? How can
>   you tell from the row count?
> - The rate source produced rows the whole time the query was stopped. Where did they go?

In [ ]:
# TO DO: start the query with a processingTime trigger


# TO DO: await 4 batches, show progress, then STOP the query

## Step 4 — Output modes

**TO DO**

Build a running count of events by `event_type` and write it to `agg_table`.

1. `groupBy("event_type").count()` on the rate stream.
2. Try to write it in `append` mode. Read the error.
3. Write it in `complete` mode instead, with `agg_checkpoint`.
4. Let it run a few batches, stop it, and query the result.

**The three modes**

- `append` — only rows that will never change again are emitted. Illegal for an
  unwatermarked aggregation, because any group could still change.
- `complete` — the entire result table is rewritten every batch. Fine for a handful of
  groups, catastrophic for millions.
- `update` — only rows that changed this batch are emitted. Needs a sink that can handle
  upserts.

> **Questions:**
> - Your aggregation has four groups, so `complete` is fine. At what point does it stop
>   being fine, and what would you switch to?
> - Step 3 wrote raw events in `append` mode with no watermark and no complaint. Why is an
>   aggregation different?

In [ ]:
# TO DO: aggregate by event_type


# TO DO: try append mode and read the error


# TO DO: write in complete mode, run a few batches, stop, inspect

## Step 5 — Break it and restart it

**TO DO**

1. Start the Step 3 query again with a `processingTime` trigger.
2. Let two batches complete, then call `query.stop()` — this simulates a cluster dying.
3. Note the row count.
4. Start the **identical** query again with the **same** checkpoint.
5. Let two more batches complete, stop it, and check the count again.

> **Questions:**
> - Look inside the checkpoint directory: `utils.list_files(rate_checkpoint)` and the
>   `offsets` folder under it. What is stored there, and how does it make the restart safe?
> - If you deleted the checkpoint before restarting, what would happen — and would you get
>   duplicates, gaps, or both?
> - Structured Streaming promises exactly-once delivery to a Delta sink. Which part of the
>   system provides the "once", the source or the sink?

In [ ]:
# TO DO: start, stop after 2 batches, note the count


# TO DO: restart with the same checkpoint, run 2 more batches, compare


# TO DO: inspect the checkpoint contents

## Before you finish

In [ ]:
streaming_utils.stop_all_streams()

## Checks

In [ ]:
from helpers import test_runner

test_runner.run("S01-streaming-fundamentals")

## Recap

- A stream is an unbounded table you can only ever see a prefix of. That single idea
  explains why `count()` fails and why `append` mode is restricted.
- The checkpoint holds the offsets. It, not your code, is what makes a restart safe.
- `availableNow` for development and backfills; `processingTime` for a query that must keep
  running. Always know which one you started.
- Complete mode rewrites everything, every batch. It is a convenience, not a default.